In [1]:
!pip install -qU langchain-ollama langchain-chroma langchain-text-splitters langchain pypdf

In [2]:
import sys
!{sys.executable} -m pip install -qU langchain

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [7]:
PDF_PATH = "./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf" 
PERSIST_DIRECTORY = "./local_chroma_db"
EMBEDDING_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2"

In [8]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [9]:
embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY,
    collection_name="rag_collection"
)

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# --- 4. Define Generation Chain ---
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
llm = OllamaLLM(model=LLM_MODEL, temperature=0.1)

system_prompt = (
    "Use the following context to answer the question. "
    "If the context does not contain the answer, say 'I don't have enough context.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

# Only pipe the prompt, model, and string parser
generation_chain = prompt | llm | StrOutputParser()


# --- 5. Step-by-Step Query Execution ---
print("RAG System is ready! (Type 'exit' or 'quit' to stop)\n")

while True:
    # This creates a text box in Jupyter for you to type your query
    query = input("Enter your question: ")
    
    # Check if the user wants to stop
    if query.lower() in ['exit', 'quit']:
        print("Exiting chat.")
        break
        
    print(f"\nThinking about: '{query}'...")
    
    # Step A: Retrieve the raw document chunks explicitly
    retrieved_docs = retriever.invoke(query)

    # Step B: Format the chunks into a single text block
    formatted_context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Step C: Feed the formatted context and query to the LLM
    answer = generation_chain.invoke({
        "context": formatted_context,
        "question": query
    })

    print("\n--- Answer ---")
    print(answer)

    print("\n--- Source Documents ---")
    # Print the metadata from the retrieved chunks to verify attribution
    for doc in retrieved_docs:
        print(f"- Source: {doc.metadata.get('source', 'Unknown')}, Page: {doc.metadata.get('page', 'Unknown')}")
    
    print("\n" + "="*50 + "\n") # Divider for the next question

RAG System is ready! (Type 'exit' or 'quit' to stop)



Enter your question:  What is robot?



Thinking about: 'What is robot?'...

--- Answer ---
I don't have enough context to answer the question "What is robot?" based on the provided references. The references appear to be related to robotics and robotics research, but they do not provide a clear definition of what a robot is.

--- Source Documents ---
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 13
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 1
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 13
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 13
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.

Enter your question:  What is Taccel



Thinking about: 'What is Taccel'...

--- Answer ---
Taccel is a simulator for tactile robotics research. It is a comprehensive tool that combines advanced solid material simulation techniques (Inverted Dynamics and Physics Constraints, or IPC) with artificial body dynamics (ABD) to simulate robots and tactile sensors in a unified framework. This allows for precise and efficient simulation of robotic manipulation tasks, including grasping and manipulation of rigid and soft objects.

--- Source Documents ---
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 24
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 5
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 0
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robot

...


Thinking about: 'What is this project about?'...

--- Answer ---
I don't have enough context to answer the question.

--- Source Documents ---
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 20
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 21
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 19
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 12
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 13




Enter your question:  Explain this topic



Thinking about: 'Explain this topic'...

--- Answer ---
The topic "Human" is not explicitly mentioned in the provided context. The context only discusses guidelines for research papers, specifically regarding crowdsourcing, LLMs, theoretical results, experimental result reproducibility, and societal impact.

Therefore, I don't have enough context to explain the topic "Human".

--- Source Documents ---
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 20
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 21
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 16
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 12
- Source: ./NeurIPS-2025-taccel-sc

Enter your question:  describe anything about the paper



Thinking about: 'describe anything about the paper'...

--- Answer ---
Based on the provided context, the paper appears to be related to experimental results reproducibility, specifically in the context of machine learning or artificial intelligence. The paper seems to be well-structured and provides necessary information for reproducing the main experimental results, including code and data. The authors also mention that the codebase will be released upon acceptance to improve reproducibility.

However, I don't have enough information to provide a detailed description of the paper's content, methodology, or specific results. The context only provides guidelines and answers to specific questions about experimental results reproducibility, but it does not provide a summary or overview of the paper itself.

--- Source Documents ---
- Source: ./NeurIPS-2025-taccel-scaling-up-vision-based-tactile-robotics-via-high-performance-gpu-simulation-Paper-Conference.pdf, Page: 16
- Source: ./NeurI